# Predicting Water System Violations

In [6]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
# import geopandas as gpd

#### Loading all the data in

In [7]:
popmean = pd.read_csv('datasets/Mean_Annual_NO3_Pop_Served_County_1994-2016_FINAL.csv')
violations = pd.read_csv('datasets/Mean_Annual_NO3_Violations_County_1994-2016_FINAL.csv')
pop = pd.read_csv('datasets/SDWIS_NO3_Pop_Served_County_1994-2016_FINAL.csv')
gw = pd.read_csv('datasets/SDWIS_Percent_GW_NO3_Violations_County_1994-2016_FINAL.csv')
sw = pd.read_csv('datasets/SDWIS_Percent_SW_NO3_Violations_County_1994-2016_FINAL.csv')

#### Create a function to clean FIPS columns and make sure it's ready for merging

In [8]:
def clean_fips(df, col):
    df['FIPS'] = df[col].astype(str).str.zfill(5)
    return df

popmean = clean_fips(popmean, 'FIPS')
violations = clean_fips(violations, 'FIPS')
pop = clean_fips(pop, 'FIPS')
gw = clean_fips(gw, 'FIPS')
sw = clean_fips(sw, 'FIPS')

#### Rename for clarity

In [9]:
gw = gw.rename(columns={'Perc_Viol' : 'pct_gw_violations'})
sw = sw.rename(columns={'Perc_Viol' : 'pct_sw_violations'})

#### Merging them together

In [10]:
df_final = gw.copy()

df_final = df_final.merge(violations[['FIPS', 'mean_viol_cnty']], on='FIPS', how='left')
df_final = df_final.merge(pop[['FIPS', 'Pop_Served']], on='FIPS', how='left')
df_final = df_final.merge(gw[['FIPS', 'pct_gw_violations']], on='FIPS', how='left')
df_final = df_final.merge(sw[['FIPS', 'pct_sw_violations']], on='FIPS', how='left')


In [11]:
df_final.head()

,State_County,State,STATE_FIPS,CNTY_FIPS,COUNTY,COUNTY2,FIPS,FIPS2,pct_gw_violations_x,mean_viol_cnty,Pop_Served,pct_gw_violations_y,pct_sw_violations
0,AK_Aleutians East Borough,AK,2,13,Aleutians East Borough,Aleutians East Borough,02013,f02013,0.000000,NaN,0,0.000000,0.0
1,AK_Aleutians West Census Area,AK,2,16,Aleutians West Census Area,Aleutians West Census Area,02016,f02016,0.621118,0.043478,697,0.621118,0.0
2,AK_Anchorage Municipality,AK,2,20,Anchorage Municipality,Anchorage Municipality,02020,f02020,0.071667,0.130435,534,0.071667,0.0
3,AK_Bethel Census Area,AK,2,50,Bethel Census Area,Bethel Census Area,02050,f02050,0.000000,NaN,0,0.000000,0.0
4,AK_Bristol Bay Borough,AK,2,60,Bristol Bay Borough,Bristol Bay Borough,02060,f02060,0.000000,NaN,0,0.000000,0.0


In [12]:
df_final.isna().sum()

State_County              0
State                     0
STATE_FIPS                0
CNTY_FIPS                 0
COUNTY                    0
COUNTY2                   0
FIPS                      0
FIPS2                     0
pct_gw_violations_x       0
mean_viol_cnty         2150
Pop_Served                0
pct_gw_violations_y       0
pct_sw_violations         0
dtype: int64

#### Dropped extra gw column (not shown) and renamed remaining

In [13]:
df_final = df_final.rename(columns={'pct_gw_violations_x' : 'pct_gw_violations'})
df_final = df_final[df_final['Pop_Served'] > 0]

#### Fill na values with zero for modeling

In [14]:
df_final['mean_viol_cnty'] = df_final['mean_viol_cnty'].fillna(0)

#### Create a column for violations per 1000, num of violatioms adjusted for population

In [15]:
df_final['violations_per_1000'] = (
    df_final['mean_viol_cnty'] / df_final['Pop_Served'] * 1000
)

#### Turn into categories for classification

In [16]:
df_final['high_violation'] = (
    df_final['violations_per_1000'] > df_final['violations_per_1000'].median()
).astype(int)

In [17]:
df_final.dtypes

State_County            object
State                   object
STATE_FIPS               int64
CNTY_FIPS                int64
COUNTY                  object
COUNTY2                 object
FIPS                    object
FIPS2                   object
pct_gw_violations      float64
mean_viol_cnty         float64
Pop_Served               int64
pct_gw_violations_y    float64
pct_sw_violations      float64
violations_per_1000    float64
high_violation           int64
dtype: object

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

X = df_final[['Pop_Served', 'pct_gw_violations', 'pct_sw_violations']]
y = df_final['high_violation']
X_train, X_test, y_train, y_test = train_test_split(X, y)

clf = GradientBoostingClassifier()
clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.8352059925093633

In [19]:
clf.feature_importances_

array([0.85041388, 0.12913743, 0.02044868])